In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from database_connection import load_data

In [6]:
df = load_data()

df.head()

,id,response_time,gender,return_plan,reason
0,1,2026-02-07 15:30:13,Male,Not Sure,Better quality of life abroad\r
1,2,2026-02-07 15:33:18,Male,Not Sure,Political situation\r
2,3,2026-02-07 15:34:42,Female,Not Sure,Better job opportunities abroad\r
3,4,2026-02-07 15:45:12,Female,No,Political situation\r
4,5,2026-02-07 15:45:38,Female,No,Family reasons\r


In [8]:
import pandas as pd
import numpy as np

# Binary target
df['return_binary'] = df['return_plan'].apply(lambda x: 1 if x == "Yes" else 0)

# Clean reason text
df['reason'] = df['reason'].str.strip()

# Group reasons
def categorize_reason(reason):
    if "Political" in reason:
        return "Political"
    elif "job" in reason or "salary" in reason or "quality" in reason:
        return "Economic"
    elif "study" in reason:
        return "Education"
    elif "Family" in reason:
        return "Family"
    else:
        return "Other"

df['reason_group'] = df['reason'].apply(categorize_reason)

# Check distributions
print("Return Binary Distribution:")
print(df['return_binary'].value_counts())

print("\nReason Group Distribution:")
print(df['reason_group'].value_counts())

Return Binary Distribution:
return_binary
0    64
1    18
Name: count, dtype: int64

Reason Group Distribution:
reason_group
Economic     32
Political    19
Other        15
Education    11
Family        5
Name: count, dtype: int64


In [9]:
# Merge Family into Other
df['reason_group'] = df['reason_group'].replace("Family", "Other")

print(df['reason_group'].value_counts())

reason_group
Economic     32
Other        20
Political    19
Education    11
Name: count, dtype: int64


In [10]:
import pandas as pd
import numpy as np

#Binary target (Yes = 1, No + Not Sure = 0)
df['return_binary'] = df['return_plan'].apply(lambda x: 1 if x == "Yes" else 0)

df['reason'] = df['reason'].str.strip()

def categorize_reason(reason):
    if "Political" in reason:
        return "Political"
    elif "job" in reason or "salary" in reason or "quality" in reason:
        return "Economic"
    elif "study" in reason:
        return "Education"
    else:
        return "Other"   # Family + anything else

df['reason_group'] = df['reason'].apply(categorize_reason)

print("Return Binary Distribution:")
print(df['return_binary'].value_counts())

print("\nReason Group Distribution:")
print(df['reason_group'].value_counts())

Return Binary Distribution:
return_binary
0    64
1    18
Name: count, dtype: int64

Reason Group Distribution:
reason_group
Economic     32
Other        20
Political    19
Education    11
Name: count, dtype: int64


In [13]:
pd.crosstab(df['reason_group'], df['return_binary'])

return_binary,0,1
reason_group,,
Economic,32,0
Education,10,1
Family,3,2
Other,0,15
Political,19,0


In [14]:
pd.crosstab(df['gender'], df['return_binary'])

return_binary,0,1
gender,,
Female,34,8
Male,30,10


In [15]:
def categorize_reason(reason):
    if "Political" in reason:
        return "Political"
    elif "job" in reason or "salary" in reason or "quality" in reason:
        return "Economic"
    else:
        return "Other"

df['reason_group'] = df['reason'].apply(categorize_reason)

In [16]:
print(pd.crosstab(df['reason_group'], df['return_binary']))

return_binary   0   1
reason_group         
Economic       32   0
Other          13  18
Political      19   0


In [17]:
def categorize_reason(reason):
    if "Political" in reason or "job" in reason or "salary" in reason or "quality" in reason:
        return "Structural"   # combine Political + Economic
    else:
        return "Other"

df['reason_group'] = df['reason'].apply(categorize_reason)

In [18]:
pd.crosstab(df['reason_group'], df['return_binary'])

return_binary,0,1
reason_group,,
Other,13,18
Structural,51,0


In [19]:
import statsmodels.api as sm

X = pd.get_dummies(df[['gender', 'reason_group']], drop_first=True)
X = X.astype(float)
X = sm.add_constant(X)

y = df['return_binary']

model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.245486
         Iterations 30
                           Logit Regression Results                           
Dep. Variable:          return_binary   No. Observations:                   82
Model:                          Logit   Df Residuals:                       79
Method:                           MLE   Df Model:                            2
Date:                Fri, 27 Feb 2026   Pseudo R-squ.:                  0.5336
Time:                        14:04:37   Log-Likelihood:                -20.130
converged:                       True   LL-Null:                       -43.156
Covariance Type:            nonrobust   LLR p-value:                 9.999e-11
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.1178      0.486     -0.242      0.808      -1.070  

In [20]:
import statsmodels.api as sm

# Only gender
X = pd.get_dummies(df[['gender']], drop_first=True)
X = X.astype(float)
X = sm.add_constant(X)

y = df['return_binary']

model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.523704
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:          return_binary   No. Observations:                   82
Model:                          Logit   Df Residuals:                       80
Method:                           MLE   Df Model:                            1
Date:                Fri, 27 Feb 2026   Pseudo R-squ.:                0.004913
Time:                        14:23:53   Log-Likelihood:                -42.944
converged:                       True   LL-Null:                       -43.156
Covariance Type:            nonrobust   LLR p-value:                    0.5149
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -1.4469      0.393     -3.682      0.000      -2.217      -0.677
gender_Male     0.3483    